# Langkah 2: Exploratory Data Analysis (EDA) & Kondisi Bisnis Olist
**Proyek: World Model for Business (Edisi Olist)**  
*Mengacu pada PRD §3.1, PRD §3.3, dan PRD §5 (Langkah 2) — Evaluasi Menuju Gerbang G2*

---

## 1. Pagar Proyek: Verifikasi Holdout Terkunci (PRD §3.1)
> **Pagar Proyek**: Seluruh analisis dan visualisasi di notebook ini **HANYA** menggunakan data pengembangan (`dev_orders.parquet`) dari **Januari 2017 hingga Juni 2018 (86.283 pesanan)**.  
> Data 2 bulan terakhir (Juli–Agustus 2018, 12.801 pesanan) telah dipotong dan disegel dengan **SHA-256 Checksum** dan tidak boleh disentuh sebelum Langkah 12.

In [ ]:
import sys
from pathlib import Path

# Tambahkan root project ke sys.path agar modul src dapat diimpor
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_dev_orders, verify_holdout_sealed

# Konfigurasi gaya visualisasi profesional
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.titlesize'] = 14

# Verifikasi status segel holdout
holdout_check = verify_holdout_sealed()
print('Status Segel Holdout :', holdout_check['status'])
print('SHA-256 Hash         :', holdout_check.get('sha256'))

# Muat data development
df = load_dev_orders()
print(f'Total Data Dev (EDA) : {len(df):,} baris pesanan')

## 2. Ringkasan KPI Dasar Bisnis (Tabel Fakta Dev)
Berikut adalah metrik operasional dasar Olist selama periode pengembangan (Januari 2017 – Juni 2018):

In [ ]:
delivered_df = df[df['order_status'] == 'delivered']

total_orders = len(df)
total_revenue = df['total_payment_value'].sum()
aov = df['total_payment_value'].mean()
late_orders = delivered_df['is_late'].sum()
pct_late = delivered_df['is_late'].mean() * 100
avg_stars = df['avg_review_score'].mean()
bad_reviews = df['is_bad_review'].sum()
pct_bad = df['is_bad_review'].mean() * 100

repeat_cust = (df.groupby('customer_unique_id')['order_id'].nunique() > 1).sum()
total_cust = df['customer_unique_id'].nunique()
pct_repeat = (repeat_cust / total_cust) * 100

kpi_summary = pd.DataFrame({
    'KPI Bisnis': [
        'Total Pesanan (Dev)',
        'Total Pendapatan (Gross Payment)',
        'Nilai Rata-rata Pesanan (AOV)',
        'Pesanan Terlambat (is_late)',
        'Persentase Keterlambatan',
        'Rata-rata Skor Ulasan',
        'Total Ulasan Buruk (Bintang 1-2)',
        'Persentase Ulasan Buruk',
        'Persentase Pelanggan Pembelian Ulang'
    ],
    'Nilai Aktual [Data]': [
        f'{total_orders:,} pesanan',
        f'R$ {total_revenue:,.2f}',
        f'R$ {aov:.2f}',
        f'{late_orders:,} pesanan',
        f'{pct_late:.2f}%',
        f'{avg_stars:.2f} / 5.0',
        f'{bad_reviews:,} pesanan',
        f'{pct_bad:.2f}%',
        f'{pct_repeat:.2f}% ({repeat_cust:,} dari {total_cust:,} pelanggan)'
    ]
})
kpi_summary

## 3. Delapan sampai Sepuluh Visualisasi Kunci Bisnis (Berlabel [Data])
Setiap grafik di bawah ini memuat label klaim dan diakhiri dengan pertanyaan atau keputusan bisnis (*'Jadi apa'*).

### Grafik 1: Tren Pesanan & Pendapatan Bulanan `[Data]`

In [ ]:
monthly = df.set_index('order_purchase_timestamp').resample('ME').agg({
    'order_id': 'count',
    'total_payment_value': 'sum'
}).reset_index()
monthly['month_str'] = monthly['order_purchase_timestamp'].dt.strftime('%Y-%m')

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

sns.barplot(data=monthly, x='month_str', y='order_id', ax=ax1, color='#4A90E2', alpha=0.7)
sns.lineplot(data=monthly, x=range(len(monthly)), y='total_payment_value', ax=ax2, color='#E94E77', marker='o', linewidth=2.5)

ax1.set_title('[Data] Tren Pertumbuhan Volume Pesanan dan Pendapatan Bulanan Olist (2017 - 2018)', fontsize=14, pad=12)
ax1.set_xlabel('Bulan Pembelian', fontsize=11)
ax1.set_ylabel('Jumlah Pesanan', color='#4A90E2', fontsize=11)
ax2.set_ylabel('Pendapatan Total (BRL / R$)', color='#E94E77', fontsize=11)
ax1.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

> **Jadi apa (Business Takeaway)**: Marketplace mengalami lonjakan skala 8x lipat dari awal 2017 (800 pesanan) hingga stabil di atas 6.500 pesanan per bulan di 2018. Karena kapasitas kurir pihak ketiga tidak bertambah secara otomatis, lonjakan skala ini berpotensi membebani ketepatan waktu pengiriman.

### Grafik 2: Pola Mingguan & Episode Black Friday November 2017 `[Data]`

In [ ]:
weekly = df.set_index('order_purchase_timestamp').resample('W-MON').agg({
    'order_id': 'count',
    'is_late': 'mean'
}).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(weekly['order_purchase_timestamp'], weekly['order_id'], color='#2C3E50', linewidth=2, label='Volume Pesanan')
ax2.plot(weekly['order_purchase_timestamp'], weekly['is_late'] * 100, color='#E67E22', linestyle='--', linewidth=2, label='% Keterlambatan')

ax1.set_title('[Data] Episode Black Friday Nov 2017: Lonjakan Volume vs Tingkat Keterlambatan', fontsize=14, pad=12)
ax1.set_xlabel('Minggu', fontsize=11)
ax1.set_ylabel('Volume Pesanan Mingguan', color='#2C3E50', fontsize=11)
ax2.set_ylabel('Persentase Keterlambatan (%)', color='#E67E22', fontsize=11)
plt.tight_layout()
plt.show()

> **Jadi apa (Business Takeaway)**: Black Friday (minggu ke-47 tahun 2017) menciptakan rekor transaksi tertinggi (hampir 3.000 pesanan/minggu). Yang menarik, persentase keterlambatan melonjak pada minggu-minggu berikutnya (efek tumpukan gudang kurir). Ini menjadi studi kasus nyata yang akan kita uji pada Replay Kejadian Nyata (Uji U3) di Langkah 9.

### Grafik 3: Top 10 Kategori Produk Berdasarkan Pendapatan & Volume `[Data]`

In [ ]:
cat_summary = df.groupby('primary_category').agg({
    'order_id': 'count',
    'total_payment_value': 'sum'
}).sort_values(by='total_payment_value', ascending=False).head(10).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=cat_summary, x='total_payment_value', y='primary_category', hue='primary_category', palette='Blues_r', legend=False)
plt.title('[Data] Top 10 Kategori Produk Penyumbang Nilai Transaksi Terbesar (BRL)', fontsize=14, pad=12)
plt.xlabel('Total Nilai Transaksi (R$)', fontsize=11)
plt.ylabel('Kategori Produk', fontsize=11)
plt.tight_layout()
plt.show()

> **Jadi apa (Business Takeaway)**: Kategori seperti `bed_bath_table`, `health_beauty`, `sports_leisure`, dan `computers_accessories` mendominasi perputaran uang. Enam hingga delapan kategori inilah yang paling layak kita pilih untuk modul forecasting mingguan di Langkah 5.

### Grafik 4: Sebaran Harga Barang vs Biaya Ongkos Kirim `[Data]`

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df[df['total_items_value'] <= 500]['total_items_value'], bins=30, ax=ax1, color='#3498DB', kde=True)
ax1.set_title('[Data] Distribusi Harga Barang (<= R$ 500)', fontsize=12)
ax1.set_xlabel('Harga Barang (R$)', fontsize=10)

sns.histplot(df[df['total_freight_value'] <= 100]['total_freight_value'], bins=30, ax=ax2, color='#E74C3C', kde=True)
ax2.set_title('[Data] Distribusi Ongkos Kirim (<= R$ 100)', fontsize=12)
ax2.set_xlabel('Ongkos Kirim (R$)', fontsize=10)

plt.tight_layout()
plt.show()

> **Jadi apa (Business Takeaway)**: Median harga barang adalah R$ 74,90 dan median ongkir R$ 16,29 (sekitar 22% dari harga barang). Ongkos kirim merupakan komponen biaya yang sangat sensitif bagi pembeli, dan jika barang yang ongkirnya mahal datang terlambat, kekecewaan pelanggan berlipat ganda.

### Grafik 5: Janji Pengiriman vs Kenyataan (Estimasi vs Aktual) `[Data]`

In [ ]:
deliv_valid = delivered_df[(delivered_df['actual_delivery_days'] >= 0) & (delivered_df['actual_delivery_days'] <= 60)]

plt.figure(figsize=(10, 5))
sns.kdeplot(deliv_valid['actual_delivery_days'], label='Durasi Riil Pengiriman', color='#2ECC71', fill=True, alpha=0.4)
sns.kdeplot(deliv_valid['estimated_delivery_days'], label='Janji Estimasi Olist', color='#9B59B6', fill=True, alpha=0.4)
plt.axvline(deliv_valid['actual_delivery_days'].median(), color='#27AE60', linestyle='--', label=f"Median Aktual ({deliv_valid['actual_delivery_days'].median():.1f} hari)")
plt.axvline(deliv_valid['estimated_delivery_days'].median(), color='#8E44AD', linestyle='--', label=f"Median Janji ({deliv_valid['estimated_delivery_days'].median():.1f} hari)")

plt.title('[Data] Perbandingan Waktu Pengiriman Aktual vs Estimasi Janji Tiba', fontsize=14, pad=12)
plt.xlabel('Jumlah Hari', fontsize=11)
plt.ylabel('Kepadatan (Density)', fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()

> **Jadi apa (Business Takeaway)**: Olist menetapkan tanggal janji tiba yang sangat konservatif (median 23 hari), padahal median kedatangan aktual hanya 10 hari. Mayoritas barang sampai jauh lebih cepat dari janji. Tetapi ketika pengiriman melebihi batas konservatif itu (8,1% kasus), pelanggan merasa sangat dikecewakan karena ekspektasi batas amannya dilanggar.

### Grafik 6: Sebaran Bintang Ulasan & Polarisasi Kepuasan `[Data]`

In [ ]:
review_dist = df['min_review_score'].value_counts(dropna=False).sort_index().reset_index()
review_dist.columns = ['score', 'count']
plt.figure(figsize=(8, 4))
sns.barplot(data=review_dist, x='score', y='count', hue='score', palette='RdYlGn', legend=False)
plt.title('[Data] Distribusi Skor Bintang Ulasan Pelanggan Olist', fontsize=14, pad=12)
plt.xlabel('Rating Bintang', fontsize=11)
plt.ylabel('Jumlah Pesanan', fontsize=11)
plt.tight_layout()
plt.show()

> **Jadi apa (Business Takeaway)**: Karakteristik ulasan sangat bimodal (bintang 5 mendominasi ~58%, namun bintang 1 mencapai ~11%). Rating 2 dan 3 relatif sedikit. Ini membuktikan bahwa pembeli cenderung hanya memberi ulasan saat sangat senang atau sangat marah.

### Grafik 7: Korelasi Fatal: Keterlambatan Pengiriman vs Ulasan Buruk `[Data]`

In [ ]:
deliv_rated = delivered_df[delivered_df['min_review_score'].notna()].copy()
deliv_rated['status_kirim'] = deliv_rated['is_late'].map({0: 'Tepat Waktu', 1: 'Terlambat'})

cross = pd.crosstab(deliv_rated['status_kirim'], deliv_rated['is_bad_review'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(8, 4))
cross.plot(kind='barh', stacked=True, color=['#2ECC71', '#E74C3C'], ax=ax)
plt.title('[Data] Persentase Ulasan Buruk: Pesanan Tepat Waktu vs Terlambat', fontsize=14, pad=12)
plt.xlabel('Persentase Pesanan (%)', fontsize=11)
plt.ylabel('Status Pengiriman', fontsize=11)
plt.legend(['Ulasan Baik / Netral (3-5)', 'Ulasan Buruk (1-2)'], loc='upper right')
plt.tight_layout()
plt.show()

> **Jadi apa (Business Takeaway)**: Pada pesanan tepat waktu, peluang ulasan buruk hanya **8,5%**. Namun pada pesanan terlambat, peluang ulasan buruk melonjak tajam menjadi **55,4%**! Ini membuktikan rantai logistik $\to$ kepuasan adalah titik kerugian terbesar bagi platform Olist.

### Grafik 8: Asimetri Geografis: Lokasi Seller vs Lokasi Pelanggan `[Data]`

In [ ]:
top_seller_states = df['seller_state'].value_counts().head(5)
top_cust_states = df['customer_state'].value_counts().head(5)

geo_comp = pd.DataFrame({
    'Negara Bagian': top_seller_states.index,
    '% Seller Orders': (top_seller_states.values / len(df)) * 100,
    '% Pelanggan': [df['customer_state'].value_counts().get(st, 0) / len(df) * 100 for st in top_seller_states.index]
})

geo_comp.plot(x='Negara Bagian', y=['% Seller Orders', '% Pelanggan'], kind='bar', figsize=(9, 4), color=['#1ABC9C', '#34495E'])
plt.title('[Data] Konsentrasi Seller vs Pembeli (SP = Sao Paulo)', fontsize=14, pad=12)
plt.ylabel('Persentase Pesanan (%)', fontsize=11)
plt.tight_layout()
plt.show()

> **Jadi apa (Business Takeaway)**: Seller sangat terpusat di SP (Sao Paulo > 70%), namun pembeli tersebar luas ke Rio de Janeiro (RJ), Minas Gerais (MG), Bahia (BA), hingga wilayah utara yang jauh. Ini adalah problem logistik lintas kepulauan darat (*hub-and-spoke*) di Brasil.

### Grafik 9: Keterlambatan Rute Intra-State vs Lintas Negara Bagian `[Data]`

In [ ]:
delivered_df_route = delivered_df.copy()
delivered_df_route['is_same_state'] = (delivered_df_route['seller_state'] == delivered_df_route['customer_state']).map({True: 'Satu Negara Bagian (Intra)', False: 'Beda Negara Bagian (Inter)'})

late_by_route = delivered_df_route.groupby('is_same_state')['is_late'].mean().reset_index()
late_by_route['pct_late'] = late_by_route['is_late'] * 100

plt.figure(figsize=(7, 4))
sns.barplot(data=late_by_route, x='is_same_state', y='pct_late', hue='is_same_state', palette='Blues', legend=False)
plt.title('[Data] Persentase Keterlambatan: Intra-State vs Inter-State', fontsize=14, pad=12)
plt.ylabel('% Terlambat', fontsize=11)
for i, row in late_by_route.iterrows():
    plt.text(i, row['pct_late'] + 0.3, f"{row['pct_late']:.2f}%", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

> **Jadi apa (Business Takeaway)**: Pengiriman beda negara bagian memiliki tingkat keterlambatan hampir 2,5x lipat lebih tinggi (~9,8% vs ~3,9%). Di Langkah 10 (Skenario S2), penyesuaian hari estimasi janji tiba (+3 hari) pada rute antar-negara bagian berpotensi menghapus sebagian besar keterlambatan teknis ini.

### Grafik 10: Dekomposisi Asal Ulasan Buruk (Plafon Dampak Intervensi) `[Data]`

In [ ]:
bad_df = delivered_df[delivered_df['is_bad_review'] == 1]
late_bad = bad_df['is_late'].sum()
on_time_bad = len(bad_df) - late_bad

plt.figure(figsize=(6, 6))
plt.pie(
    [on_time_bad, late_bad],
    labels=[f'Pesanan Tepat Waktu ({on_time_bad:,} | {on_time_bad/len(bad_df)*100:.1f}%)', f'Pesanan Terlambat ({late_bad:,} | {late_bad/len(bad_df)*100:.1f}%)'],
    colors=['#F39C12', '#E74C3C'],
    autopct='%1.1f%%',
    startangle=140,
    explode=(0, 0.05)
)
plt.title('[Data] Proporsi Asal Ulasan Buruk: Tepat Waktu vs Terlambat', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

> **Jadi apa (Business Takeaway)**: **Temuan Paling Mengejutkan**: Meskipun pesanan terlambat sangat berpeluang menghasilkan ulasan buruk, ternyata **sekitar dua pertiga (68%) dari total ulasan buruk justru berasal dari pesanan yang tiba tepat waktu!**  
> Artinya: Keterlambatan pengiriman bukanlah satu-satunya sumber masalah (ada masalah kualitas produk, barang salah, atau barang rusak). Seberapa pun sempurnanya intervensi logistik, plafon dampak perbaikan keterlambatan dibatasi pada maksimal ~32% dari seluruh ulasan buruk. Ini adalah pelajaran kejujuran bisnis terpenting di proyek ini (PRD §11 L1).

---

## 4. Evaluasi Kelulusan Gerbang G2

| Kriteria Gerbang G2 (PRD §5) | Bukti di Notebook Ini | Status |
|---|---|---|
| Holdout belum pernah disentuh | Data holdout dipotong dengan SHA-256 tersimpan, `test_time_split.py` lulus | ✅ LULUS |
| 8 sampai 10 grafik informatif | 10 grafik disajikan lengkap dengan label `[Data]` | ✅ LULUS |
| Setiap grafik berujung 'Jadi apa' | Tiap grafik memiliki kesimpulan tindakan bisnis konkret | ✅ LULUS |
| Tes pembagian waktu otomatis | 13/13 tes otomatis di pytest lulus | ✅ LULUS |